# Baseline geométrico: rasgos a mano + HistGradientBoosting

La referencia honesta para decir cuánto aporta Sonata. **Sin red neuronal ni GPU**: 20 rasgos
geométricos por voxel de 0,10 m (PCA local a 0,4 / 1,2 / 3 m, altura sobre el suelo, altura y
ocupación de la columna de 1 m, densidad, intensidad, color; los calcula `codigo/postes/preparar_datos.py` y
quedan en `datos/intermedio/cache/geometrica/`) y un *HistGradientBoosting* de **tres clases** (fondo, `poste_bt`,
`poste_mt`) con peso por clase `√(fondo/clase)`.

Misma partición (las carpetas `train/` y `test/` de `datos/crudo/etiquetado/`, validación apartada de train),
misma detección por objeto, mismo filtro de forma y misma clase por objeto que el cuaderno de
Sonata: todo eso vive en `evaluacion_postes.py` y `postes_datos.py`. Las rejillas difieren
(0,10 m aquí, 0,08 m Sonata), así que la comparación es por **objeto**, que no depende del voxel.

> Corre en cualquier kernel con numpy, scipy, scikit-learn, laspy y pyproj; el `py312-sonata`
> sirve. Entrenar tarda del orden de un minuto.

In [ ]:
import os, sys, json, time, subprocess, pickle
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "etapa_4_5_modelado_evaluacion").is_dir())
AQUI = RAIZ / "etapa_4_5_modelado_evaluacion" / "codigo" / "postes"
os.chdir(AQUI); sys.path.insert(0, str(AQUI))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, log_loss

import postes_datos as pdt
from postes_datos import cargar_cache, contar_postes
from evaluacion_postes import (barrido_umbral, elegir_umbral, graficar_umbral, evaluar_conjunto,
                               escribir_salidas, comparar_resumenes)

SEMILLA = 0
PRUEBA_RAPIDA = os.environ.get("POSTES_PRUEBA_RAPIDA") == "1"
SALIDA_BASE = RAIZ / "etapa_4_5_modelado_evaluacion" / "resultados" / ("postes_prueba_rapida" if PRUEBA_RAPIDA else "postes")
VENV_SONATA = Path(r"F:\venvs\sonata\Scripts\python.exe")
PY = str(VENV_SONATA) if VENV_SONATA.exists() else sys.executable
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
print(f"kernel {sys.executable}\n" + ("prueba rapida" if PRUEBA_RAPIDA else "modo completo"))

## Parámetros

La partición la dicen las carpetas, igual que en el cuaderno de Sonata (ver `postes_datos.
particion_por_carpetas`). Los hiperparámetros del HGB son los del baseline del 30-oct; no se
afinaron porque su papel es ser referencia, no ganar.

In [ ]:
# "BT" o "MT": un modelo de una clase, como sonata_mlp. None: las dos clases a la vez, que
# solo vale si cada bloque esta en el mismo conjunto para BT y para MT (si no, falla y dice cuales).
CLASE_OBJETIVO = "BT"
# 10-nov-2026: la particion es POR CLASE. El modelo BT la toma de
# datos/crudo/etiquetado/<escaneo>/poste_bt/<train|test>/ y el MT de .../poste_mt/<train|test>/;
# un bloque puede estar en train para una clase y en test para la otra.
VAL_DE_TRAIN = (list(pdt.VAL_DE_TRAIN[CLASE_OBJETIVO]) if CLASE_OBJETIVO
                else sorted(set(sum(pdt.VAL_DE_TRAIN.values(), []))))
PARTICION = pdt.particion_por_carpetas(CLASE_OBJETIVO, VAL_DE_TRAIN)
if PRUEBA_RAPIDA:
    PARTICION = {"BT": {"train": ["AVP01MT2611202501_b_1465_43337", "PACHAP01MT0108202501_b_1494_43242"],
                        "val": ["PACHAP01MT0108202501_b_1497_43238"], "test": ["PACHAP01MT0108202501_b_1494_43240"]},
                 "MT": {"train": ["AVP01MT2611202501_b_1465_43337"],
                        "val": ["PACHAP01MT0108202502_b_1489_43238"], "test": ["PACHAP01MT0108202502_b_1489_43239"]},
                 }[CLASE_OBJETIVO or "BT"]
for k, v in PARTICION.items():
    assert v, (f"la particion '{k}' esta vacia: revisa datos/crudo/etiquetado/<escaneo>/poste_{(CLASE_OBJETIVO or 'bt').lower()}/"
               "train y test, y VAL_DE_TRAIN")

MAX_FONDO_TRAIN = 400_000       # voxeles de fondo por bloque (los de poste entran todos)
MAX_FONDO_VAL   = 200_000
HGB = dict(max_iter=50 if PRUEBA_RAPIDA else 300, learning_rate=0.1, max_leaf_nodes=31)
UMBRALES        = np.round(np.concatenate([[0.01, 0.02, 0.03], np.arange(0.05, 0.96, 0.05)]), 2)
ETIQUETA_MODELO = f"geo_hgb_solo_{CLASE_OBJETIVO.lower()}" if CLASE_OBJETIVO else "geo_hgb"
# cada corrida en su carpeta: resultados/postes/corrida<N>_<fecha>_<modelo>/
SALIDA = pdt.nueva_corrida(ETIQUETA_MODELO, SALIDA_BASE, fija=PRUEBA_RAPIDA)
print("salidas ->", SALIDA.relative_to(RAIZ))

CLASES  = (pdt.CLASE_POR_SUFIJO[CLASE_OBJETIVO],) if CLASE_OBJETIVO else (5, 12)
NOMBRES = ["fondo"] + [pdt.NOMBRE_CLASE[c] for c in CLASES]
INDICE  = np.zeros(256, dtype="i8")
for i, c in enumerate(CLASES):
    INDICE[c] = i + 1
print(json.dumps(PARTICION, indent=1)); print("clases:", NOMBRES)

## 1 · Preparación de la cache geométrica

`codigo/postes/preparar_datos.py` calcula los rasgos de cada bloque (1-3 min por bloque, CPU) y los guarda en
`datos/intermedio/cache/geometrica/<bloque>.npz`. Salta los que ya están; si un recorte es más nuevo que su cache, recalcula
solo la etiqueta.

In [ ]:
inv = pdt.inventario()

def estado(bloque, marca):
    if not marca.exists():
        return "falta"
    fila = inv[inv["bloque"] == bloque].iloc[0].to_dict()
    return "etiquetas viejas" if pdt.etiquetas_desactualizadas(fila, marca) else "ok"

inv["geometrico"] = inv["bloque"].map(lambda b: estado(b, pdt.ruta_cache(b)))
inv["particion"] = inv["bloque"].map({b: k for k, bs in PARTICION.items() for b in bs})
usados = inv[inv["particion"].notna()]
display(usados[["bloque", "escaneo", f"conjunto_{CLASE_OBJETIVO}" if CLASE_OBJETIVO else "conjunto",
                "particion", "BT", "MT", "geometrico"]]
        .assign(BT=lambda d: d["BT"].notna(), MT=lambda d: d["MT"].notna()))

def lanzar(script, bloques):
    if not bloques:
        return
    print(f"-> {PY} {script} {' '.join(bloques)}")
    p = subprocess.Popen([PY, "-u", str(AQUI / script), *bloques], cwd=str(AQUI),
                         env={**os.environ, "PYTHONIOENCODING": "utf-8"},
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    cola = []
    for linea in p.stdout:
        if "RuntimeWarning" in linea:
            continue
        print("   " + linea.rstrip()); cola = (cola + [linea.rstrip()])[-25:]
    if p.wait() != 0:
        raise RuntimeError(f"{script} termino con codigo {p.returncode}. Ultimas lineas:\n" + "\n".join(cola))

lanzar("preparar_datos.py", usados.loc[usados["geometrico"] != "ok", "bloque"].tolist())

## 2 · Los datos

In [ ]:
filas = []
for parte, bloques in PARTICION.items():
    for b in bloques:
        d = cargar_cache(b)
        n_post = contar_postes(d["P"], d["Y"], CLASES)
        filas.append({"particion": parte, "bloque": b, "escaneo": d["escaneo"][:6], "voxeles": len(d["Y"]),
                      "vox_poste_bt": int((d["Y"] == 5).sum()), "vox_poste_mt": int((d["Y"] == 12).sum()),
                      "postes_bt": n_post.get(5, 0), "postes_mt": n_post.get(12, 0), "rasgos": d["X"].shape[1]})
tabla_datos = pd.DataFrame(filas)
display(tabla_datos)
display(tabla_datos.groupby("particion")[["voxeles", "vox_poste_bt", "vox_poste_mt", "postes_bt", "postes_mt"]].sum())
ETIQUETAS = pdt.huella_etiquetas(tabla_datos["bloque"])
print(f"etiquetas: {ETIQUETAS['n_recortes']} recortes, huella {ETIQUETAS['md5'][:12]}")

## 3 · Entrenamiento y curva de validación

Todos los voxeles de poste y hasta `MAX_FONDO_TRAIN` de fondo por bloque. La curva muestra la
pérdida logarítmica por iteración en train y en la muestra de validación, y el AP de *poste contra
fondo* en validación: si la pérdida de validación sube al final, sobran iteraciones.

In [ ]:
def cuenta(y):
    return " · ".join(f"{NOMBRES[i]} {int((y == i).sum()):,}" for i in range(len(NOMBRES)))

t0 = time.perf_counter()
def solo_clases(X, Y):
    "Con un modelo de una clase, los postes de la otra no entran (ni como poste ni como fondo)."
    m = np.isin(Y, (0,) + tuple(CLASES))
    return X[m], Y[m]

Xg, Yg, _ = pdt.conjunto(PARTICION["train"], max_fondo=MAX_FONDO_TRAIN, semilla=SEMILLA)
Xg, Yg = solo_clases(Xg, Yg)
yg = INDICE[Yg]
n = np.bincount(yg, minlength=len(NOMBRES)).astype("f8")
PESOS_CLASE = np.where(n > 0, np.sqrt(n[0] / np.maximum(n, 1)), 1.0); PESOS_CLASE[0] = 1.0
hgb = HistGradientBoostingClassifier(**HGB, early_stopping=False, random_state=SEMILLA).fit(Xg, yg, sample_weight=PESOS_CLASE[yg])
print(f"HGB entrenado con {len(yg):,} voxeles ({cuenta(yg)}) en {time.perf_counter() - t0:.0f} s · "
      "pesos " + ", ".join(f"{NOMBRES[i]} {PESOS_CLASE[i]:.1f}" for i in range(len(NOMBRES))))

Xgv, Ygv, _ = pdt.conjunto(PARTICION["val"], max_fondo=MAX_FONDO_VAL, semilla=SEMILLA + 1)
Xgv, Ygv = solo_clases(Xgv, Ygv)
ygv = INDICE[Ygv]
curva = []
for i, (ptr, pva) in enumerate(zip(hgb.staged_predict_proba(Xg[::20]), hgb.staged_predict_proba(Xgv)), 1):
    if i % 10 == 0 or i == 1:
        curva.append({"iter": i, "logloss_train": log_loss(yg[::20], ptr, labels=hgb.classes_),
                      "logloss_val": log_loss(ygv, pva, labels=hgb.classes_),
                      "ap_poste_val": average_precision_score(ygv > 0, 1 - pva[:, 0])})
curva = pd.DataFrame(curva)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(curva["iter"], curva["logloss_train"], label="train"); ax[0].plot(curva["iter"], curva["logloss_val"], label="val")
ax[0].set_title("log-loss"); ax[0].legend(); ax[0].set_xlabel("iteración")
ax[1].plot(curva["iter"], curva["ap_poste_val"], color="C2"); ax[1].set_title("AP poste vs fondo · validación"); ax[1].set_xlabel("iteración")
plt.tight_layout(); plt.savefig(SALIDA / f"curvas_{ETIQUETA_MODELO}.png"); plt.show()

def probs_geo(X):
    # columnas en el orden fondo, bt, mt aunque alguna clase no haya aparecido en train
    p = np.zeros((len(X), len(NOMBRES)), dtype="f4")
    p[:, hgb.classes_.astype(int)] = hgb.predict_proba(X)
    return p

## 4 · Umbral en validación, por objeto

In [ ]:
t0 = time.perf_counter()
datos_val = {b: cargar_cache(b) for b in PARTICION["val"]}
COMPARTIDOS_VAL = pdt.marcar_compartidos(datos_val, PARTICION["train"], cargar_cache)
print(f"postes de validacion compartidos con train, fuera de la metrica: {COMPARTIDOS_VAL}")
probs_val = {b: probs_geo(d["X"]) for b, d in datos_val.items()}
tabla_val = barrido_umbral(datos_val, probs_val, UMBRALES, CLASES)
UMBRAL = elegir_umbral(tabla_val)
print(f"umbral elegido: {UMBRAL} · {time.perf_counter() - t0:.0f} s")
display(tabla_val[tabla_val["filtro_forma"]].drop(columns="filtro_forma").round(3))
graficar_umbral(tabla_val, UMBRAL, SALIDA / f"umbral_{ETIQUETA_MODELO}.png", "· HGB geométrico")

## 5 · Test

Detección por bloque y tabla por clase, con el umbral fijado en validación. Debajo, el mismo
cálculo sobre train como diagnóstico (no es métrica: el modelo vio esos bloques).

In [ ]:
datos_test = {b: cargar_cache(b) for b in PARTICION["test"]}
COMPARTIDOS_TEST = pdt.marcar_compartidos(datos_test, PARTICION["train"] + PARTICION["val"], cargar_cache)
print(f"postes de test compartidos con train o validacion, fuera de la metrica: {COMPARTIDOS_TEST}")
probs_test = {b: probs_geo(d["X"]) for b, d in datos_test.items()}
tabla_test, tabla_test_clases, clase_test = evaluar_conjunto(datos_test, probs_test, UMBRAL, CLASES)
print(f"TEST · umbral {UMBRAL}\ndeteccion (poste o no):"); display(tabla_test)
print("por clase:"); display(tabla_test_clases)

datos_train = {b: cargar_cache(b) for b in PARTICION["train"]}
tabla_train, tabla_train_clases, _ = evaluar_conjunto(datos_train, {b: probs_geo(d["X"]) for b, d in datos_train.items()}, UMBRAL, CLASES)
del datos_train
print("TRAIN (diagnostico, no es metrica):"); display(tabla_train.tail(1)); display(tabla_train_clases)

## 6 · Salidas

`<bloque>_PRED_geo_hgb.laz`, `<bloque>_candidatos_geo_hgb.csv`, el modelo en
`hgb_geo_hgb.pkl` y `resumen_geo_hgb.json`, con el mismo esquema que el cuaderno de
Sonata para poder compararlos.

In [ ]:
cand = escribir_salidas(SALIDA, ETIQUETA_MODELO, datos_test, clase_test, probs_test, NOMBRES)
if len(cand):
    display(pd.crosstab(cand["clase_predicha"], cand["etiqueta"]))
    print(f"{len(cand)} candidatos en test, {int((cand['etiqueta'] == 'fondo').sum())} sin poste etiquetado cerca")

with open(SALIDA / f"hgb_{ETIQUETA_MODELO}.pkl", "wb") as fh:
    pickle.dump({"modelo": hgb, "umbral": UMBRAL, "clases": list(CLASES), "nombres": NOMBRES,
                 "rasgos": list(pdt.NOMBRES_RASGOS), "particion": PARTICION}, fh)

resumen = {"etiqueta_modelo": ETIQUETA_MODELO,
           "modelo": f"rasgos geometricos ({tabla_datos['rasgos'].iloc[0]}) + HistGradientBoosting de {len(NOMBRES)} clases",
           "fecha": time.strftime("%Y-%m-%d %H:%M"), "particion": PARTICION, "val_de_train": VAL_DE_TRAIN,
           "clases": NOMBRES[1:], "etiquetas": ETIQUETAS, "postes_compartidos_excluidos": {"val": COMPARTIDOS_VAL, "test": COMPARTIDOS_TEST},
           "config": {**HGB, "max_fondo_train": MAX_FONDO_TRAIN, "max_fondo_val": MAX_FONDO_VAL,
                      "pesos_clase": dict(zip(NOMBRES, PESOS_CLASE.tolist())), "semilla": SEMILLA},
           "umbral": UMBRAL, "curva": curva.round(5).to_dict("records"),
           "datos": tabla_datos.to_dict("records"),
           "validacion_umbral": tabla_val.round(4).to_dict("records"),
           "test": tabla_test.to_dict("records"), "test_por_clase": tabla_test_clases.to_dict("records"),
           "train_diagnostico": tabla_train.tail(1).to_dict("records"),
           "train_diagnostico_por_clase": tabla_train_clases.to_dict("records")}
(SALIDA / f"resumen_{ETIQUETA_MODELO}.json").write_text(json.dumps(resumen, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("->", SALIDA / f"resumen_{ETIQUETA_MODELO}.json")

## 7 · Comparación con los demás cuadernos

Una fila por corrida (`etapa_4_5_modelado_evaluacion/resultados/postes/corrida<N>_<fecha>_<modelo>/resumen_<modelo>.json`). Compara solo corridas con la misma partición y la
misma huella de etiquetas.

In [ ]:
display(comparar_resumenes(SALIDA_BASE).round(3))